# Neu — OCGEO-01 «ОЦЕНЩИК-ГЕОМЕТРИЯ», ТЕСТ (Colab)

1. **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить** (Runtime → Change runtime type → T4 GPU).
2. **Среда выполнения → Выполнить все** (Runtime → Run all).
3. Когда Colab попросит доступ к Google Диску — разрешить.

Идёт около 45–60 минут: шесть моделей Pythia, 5669 задач сложения, затем анализ по предрегистрации (коммит 1b43fe66). Вкладку не закрывать.

Результат — папка **Neu_colab/ocgeo_test** на Диске. Когда в последней ячейке появится «готово», напишите Claude «готово». Личных данных ноутбук не пишет.

In [ ]:
# 1. Видеокарта и Google Диск
import os, torch
assert torch.cuda.is_available(), 'Нет видеокарты: Среда выполнения → Сменить среду выполнения → T4 GPU, затем «Выполнить все» заново'
print('видеокарта:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Neu_colab/ocgeo_test'
os.makedirs(OUT, exist_ok=True)
print('папка результата:', OUT)

In [ ]:
# 2. Код опыта с GitHub — коммит регистрации, сверка с зарегистрированными sha256 (prereg_hashes.json)
import hashlib, urllib.request
COMMIT = '1b43fe66b489285a166a34fb7825fb7c90e7d63e'
BASE = f'https://raw.githubusercontent.com/131ymm-commits/Neu/{COMMIT}/experiments/2026-11-14_OCGEO-01'
SHA = {"ocgeo_extract_test.py": "4aed805d7b31bbd1d43a11e63fbe5206c62fe1e530a5a963b514de7d99aa99a7", "analyze_ocgeo.py": "d7724c385718ec5fade09cf385b69edd730230cd8423b2bc87274a2c9214ffca", "analyze_ocgeo_test.py": "2aee14122df8bc287fe4a49b33dafa98c76ce3c85230ece2cbd41a03c85194ca"}
for name in SHA:
    data = urllib.request.urlopen(f'{BASE}/{name}').read()
    h = hashlib.sha256(data).hexdigest()
    assert h == SHA[name], f'sha256 не совпал с регистрацией: {name}'
    open(name, 'wb').write(data)
    print(name, 'sha256 совпал с регистрацией')

In [ ]:
# 3. Ответы голов и соседи (долго: около 30–40 минут)
!pip -q install accelerate
!python ocgeo_extract_test.py

In [ ]:
# 4. Анализ по предрегистрации (вложенный бутстреп, 4000 повторов: около 10 минут)
!python analyze_ocgeo_test.py test_out test_out/analysis.json

In [ ]:
# 5. Запись на Диск
import shutil, glob
shutil.copy('test_out/analysis.json', os.path.join(OUT, 'analysis.txt'))
for f in glob.glob('test_out/*'):
    shutil.copy(f, OUT)
drive.flush_and_unmount()
print('готово: записано в Neu_colab/ocgeo_test — напишите Claude «готово»')